# Election Bloc Change Prediction Project
## Notebook 02 — Historical transition panel and CLR targets

Builds all nine consecutive transitions from K16→K17 through K24→K25. The final transition is retained but marked as locked.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 300)
REPO_URL = "https://github.com/IfatDav/Election_Bloc_Prediction_Project.git"
DEFAULT_REPO = Path("/content/Election_Bloc_Prediction_Project")

def locate_repo():
    candidates=[]
    if os.getenv("ELECTION_PROJECT_ROOT"):
        candidates.append(Path(os.environ["ELECTION_PROJECT_ROOT"]).expanduser())
    cwd=Path.cwd().resolve()
    candidates += [cwd, *cwd.parents, DEFAULT_REPO]
    for p in candidates:
        if (p/"data"/"raw").exists(): return p
    if Path("/content").exists():
        if DEFAULT_REPO.exists(): shutil.rmtree(DEFAULT_REPO)
        subprocess.run(["git","clone","--depth","1",REPO_URL,str(DEFAULT_REPO)],check=True)
        return DEFAULT_REPO
    raise FileNotFoundError("Repository not found. Set ELECTION_PROJECT_ROOT.")

REPO_ROOT=locate_repo()
RAW_DIR=REPO_ROOT/"data"/"raw"
INTERIM_DIR=REPO_ROOT/"data"/"interim"
PROCESSED_DIR=REPO_ROOT/"data"/"processed"
REPORTS_DIR=REPO_ROOT/"reports"
TABLES_DIR=REPORTS_DIR/"tables"
FIGURES_DIR=REPORTS_DIR/"figures"
SUMMARIES_DIR=REPORTS_DIR/"summaries"
MODELS_DIR=REPO_ROOT/"models"
for d in [INTERIM_DIR,PROCESSED_DIR,TABLES_DIR,FIGURES_DIR,SUMMARIES_DIR,MODELS_DIR]: d.mkdir(parents=True,exist_ok=True)
print("Repository:",REPO_ROOT)

In [ ]:
SOURCE=INTERIM_DIR/"election_bloc_results.csv"
if not SOURCE.exists() and (REPO_ROOT/".git").exists(): subprocess.run(["git","-C",str(REPO_ROOT),"pull","--ff-only","origin","main"],check=True)
results=pd.read_csv(SOURCE,dtype={"locality_symbol":"string"},low_memory=False)
MODELED_BLOCS=["Right","Center_Left","Haredi","Arab"]
required={"locality_symbol","election_number","target_election","year","valid_votes",*[f"{b}_pct" for b in MODELED_BLOCS]}
missing=required-set(results.columns)
if missing: raise KeyError(missing)
if results.duplicated(["locality_symbol","election_number"]).any(): raise ValueError("Duplicate locality-election rows")
print(results.groupby("election_number").size())

## Build consecutive-election matches

In [ ]:
frames=[]; audits=[]; unmatched=[]
for prev_n,curr_n in zip(range(16,25),range(17,26)):
    p=results.loc[results.election_number.eq(prev_n)].copy(); c=results.loc[results.election_number.eq(curr_n)].copy()
    keep=["locality_symbol","election_locality_name","year","eligible_voters","votes_cast","invalid_votes","valid_votes","turnout_pct",*[f"{b}_pct" for b in MODELED_BLOCS]]
    p=p[keep].add_prefix("previous_").rename(columns={"previous_locality_symbol":"locality_symbol"})
    c=c[keep].add_prefix("current_").rename(columns={"current_locality_symbol":"locality_symbol"})
    outer=p.merge(c,on="locality_symbol",how="outer",indicator=True)
    audits.append({"transition_id":f"K{prev_n}_to_K{curr_n}","previous_rows":len(p),"current_rows":len(c),"matched_rows":int(outer._merge.eq("both").sum()),"previous_only":int(outer._merge.eq("left_only").sum()),"current_only":int(outer._merge.eq("right_only").sum())})
    u=outer.loc[~outer._merge.eq("both"),["locality_symbol","previous_election_locality_name","current_election_locality_name","_merge"]].copy();u["transition_id"]=f"K{prev_n}_to_K{curr_n}";unmatched.append(u)
    m=outer.loc[outer._merge.eq("both")].drop(columns="_merge").copy()
    m["previous_election_number"]=prev_n;m["current_election_number"]=curr_n;m["transition_id"]=f"K{prev_n}_to_K{curr_n}";m["election_gap"]=1;m["final_test_locked"]=curr_n==25
    m["locality_name"]=m.current_election_locality_name.fillna(m.previous_election_locality_name)
    m["locality_name_changed"]=m.previous_election_locality_name.fillna("").ne(m.current_election_locality_name.fillna(""))
    frames.append(m)
panel=pd.concat(frames,ignore_index=True);match_audit=pd.DataFrame(audits);unmatched=pd.concat(unmatched,ignore_index=True)
match_audit

## Percentage-point and CLR change targets

In [ ]:
for b in MODELED_BLOCS: panel[f"delta_{b}_pct"]=panel[f"current_{b}_pct"]-panel[f"previous_{b}_pct"]
panel["turnout_delta_pct"]=panel.current_turnout_pct-panel.previous_turnout_pct
panel["valid_vote_growth_pct"]=np.where(panel.previous_valid_votes.gt(0),(panel.current_valid_votes/panel.previous_valid_votes-1)*100,np.nan)
EPS=1e-6
def clr(pct):
    x=np.clip(np.asarray(pct,float)/100,EPS,None);x=x/x.sum(axis=1,keepdims=True);l=np.log(x);return l-l.mean(axis=1,keepdims=True)
def invclr(z):
    z=np.asarray(z,float);e=np.exp(z-z.max(axis=1,keepdims=True));return e/e.sum(axis=1,keepdims=True)*100
prev_cols=[f"previous_{b}_pct" for b in MODELED_BLOCS];curr_cols=[f"current_{b}_pct" for b in MODELED_BLOCS]
pclr=clr(panel[prev_cols]); cclr=clr(panel[curr_cols]); dclr=cclr-pclr
for j,b in enumerate(MODELED_BLOCS): panel[f"previous_clr_{b}"]=pclr[:,j];panel[f"current_clr_{b}"]=cclr[:,j];panel[f"delta_clr_{b}"]=dclr[:,j]
for b in MODELED_BLOCS:
    panel[f"persistence_predicted_{b}_pct"]=panel[f"previous_{b}_pct"]
    panel[f"persistence_absolute_error_{b}_pct"]=(panel[f"current_{b}_pct"]-panel[f"previous_{b}_pct"]).abs()
panel["persistence_mean_absolute_error"]=panel[[f"persistence_absolute_error_{b}_pct" for b in MODELED_BLOCS]].mean(axis=1)
if np.abs(invclr(cclr).sum(axis=1)-100).max()>1e-8: raise ValueError("CLR inverse failed")

## Save historical panel

In [ ]:
paths={"panel":INTERIM_DIR/"election_transition_panel.csv","match":TABLES_DIR/"election_transition_match_audit.csv","unmatched":TABLES_DIR/"unmatched_localities_by_transition.csv","delta":TABLES_DIR/"bloc_delta_descriptive_summary.csv","summary":SUMMARIES_DIR/"notebook_02_summary.json"}
panel.to_csv(paths["panel"],index=False,encoding="utf-8-sig");match_audit.to_csv(paths["match"],index=False,encoding="utf-8-sig");unmatched.to_csv(paths["unmatched"],index=False,encoding="utf-8-sig")
delta=pd.concat([panel.groupby("transition_id")[f"delta_{b}_pct"].describe().assign(bloc=b) for b in MODELED_BLOCS]).reset_index();delta.to_csv(paths["delta"],index=False,encoding="utf-8-sig")
summary={"notebook":"02_panel_and_target_engineering","rows":len(panel),"transitions":match_audit.to_dict("records"),"final_test":"K24_to_K25","final_test_rows":int(panel.transition_id.eq("K24_to_K25").sum())}
paths["summary"].write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print(summary)